# Dataset EDA ? Custom CNN
This notebook inspects PlantVillage dynamically. It validates files without changing the source dataset and writes lightweight evidence under `results/custom_cnn/eda`.

In [ ]:
from pathlib import Path
import csv, hashlib, statistics, random
from collections import Counter
RANDOM_SEED=42; random.seed(RANDOM_SEED)
ROOT=Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd(); DATASET=ROOT/"PlantVillage"; OUT=ROOT/"results/custom_cnn/eda"; OUT.mkdir(parents=True,exist_ok=True)
EXTS={".jpg",".jpeg",".png",".bmp",".gif",".tif",".tiff"}
classes=sorted(p.name for p in DATASET.iterdir() if p.is_dir())
rows=[]; non=[]; bad=[]
for cls in classes:
 for p in sorted((DATASET/cls).rglob("*")):
  if not p.is_file(): continue
  if p.suffix.lower() not in EXTS: non.append(str(p)); continue
  try:
   from PIL import Image
   with Image.open(p) as im: im.verify()
   with Image.open(p) as im: w,h=im.size
  except Exception as e: bad.append((str(p),str(e))); continue
  rows.append((str(p),cls,w,h))
counts=Counter(r[1] for r in rows); print(f"Classes: {len(classes)}\n{classes}\nValid images: {len(rows)}\nExcluded/corrupt: {len(bad)}; non-image: {len(non)}")
def write(name,header,data):
 with open(OUT/name,"w",newline="",encoding="utf-8") as f: csv.writer(f).writerows([header,*data])
write("class_counts.csv",["class_name","count"],sorted(counts.items())); write("corrupted_images.csv",["filepath","error"],bad); write("non_image_files.csv",["filepath"],[(x,) for x in non])
dims=Counter((r[2],r[3]) for r in rows); write("image_dimension_summary.csv",["width","height","count"],[(w,h,n) for (w,h),n in sorted(dims.items())])
summary=[("classes",len(classes)),("valid_images",len(rows)),("minimum_class_count",min(counts.values()) if counts else 0),("maximum_class_count",max(counts.values()) if counts else 0),("mean_class_count",sum(counts.values())/len(counts) if counts else 0),("median_class_count",statistics.median(counts.values()) if counts else 0),("imbalance_ratio",max(counts.values())/min(counts.values()) if counts and min(counts.values()) else 0),("corrupted_images",len(bad)),("non_image_files",len(non))]; write("eda_summary.csv",["metric","value"],summary)
try:
 import matplotlib.pyplot as plt
 plt.figure(figsize=(12,6)); plt.bar(counts.keys(),counts.values()); plt.xticks(rotation=90); plt.tight_layout(); plt.savefig(OUT/"class_distribution.png",dpi=150); plt.close()
except ImportError: print("Plotting unavailable; rerun after installing requirements.")
